# Map expansion analysis

In [ ]:
# Load the map data from JSON file
import json
import cv2

# matplotlib 3.5.x と matplotlib-inline 0.2.x の互換性対策
import matplotlib
if not hasattr(matplotlib.rcParams, '_get'):
    matplotlib.rcParams._get = matplotlib.rcParams.__getitem__

SEED = 42
NUM_DISP = 5  # Number of segments to display
#json_path = '/mnt/hdd1/data/nuscenes/maps/expansion/singapore-onenorth.json'
json_path = '../data/debug/map_annotation/nuscenes/maps/expansion/Town05.json'
with open(json_path, 'r') as f:
    map_data = json.load(f)
print('Map data keys:', map_data.keys())

#mask_path = '/mnt/hdd1/data/nuscenes/maps/53992ee3023e5494b90c316c183be829.png'
mask_path = '../data/debug/map_annotation/nuscenes/maps/basemap_Town05.png'
# Load the map mask image
map_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
print('Map mask shape:', map_mask.shape)

# Dictionary of the fields for token indexing
polygons_dict = {poly['token']: poly for poly in map_data['polygon']}
lines_dict = {line['token']: line for line in map_data['line']}
nodes_dict = {node['token']: node for node in map_data['node']}
road_segments_dict = {rs['token']: rs for rs in map_data['road_segment']}
road_blocks_dict = {rb['token']: rb for rb in map_data['road_block']}
lanes_dict = {lane['token']: lane for lane in map_data['lane']}
ped_crossings_dict = {pc['token']: pc for pc in map_data['ped_crossing']}
traffic_light_dict = {tl['token']: tl for tl in map_data['traffic_light']}

### polygon

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

# Polygons
polygons = map_data['polygon']
print('Number of polygons:', len(polygons))
print('First polygon data:', polygons[0])

# Number of nodes per polygon
nodes_per_polygon = [len(p['exterior_node_tokens']) for p in polygons]
print('Average number of nodes per polygon:', np.mean(nodes_per_polygon))
print('Max number of nodes per polygon:', np.max(nodes_per_polygon))
print('Median number of nodes per polygon:', np.median(nodes_per_polygon))
plt.hist(nodes_per_polygon, bins=20, color='blue', alpha=0.7)
plt.title('Histogram of Number of Nodes per Polygon')
plt.xlabel('Number of Nodes')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Exterior node of the first polygon
first_polygon = polygons[0]

def _get_cropped_map(bbox, map_mask, resolution=0.1, expansion=10.0):
    """
    Crop the map mask around the bounding box.

    Args:
        bbox (list): Bounding box [xmin, xmax, ymin, ymax] in meters.
        map_mask (np.ndarray): The map mask image with shape (H, W).
        resolution (float): Resolution of the map in meters per pixel (m/px).
        expansion (float): Expansion in meters around the bounding box (meters).

    Returns:
        crop_img (np.ndarray): Cropped map image.
        extent (tuple): Extent of the cropped image in meters (xmin, xmax, ymin, ymax).
    """
    img_height, img_width = map_mask.shape
    bbox_img = [  # [xmin, xmax, ymin, ymax]
        max(0, (bbox[0] - expansion) / resolution),
        min(img_width, (bbox[1] + expansion) / resolution),
        max(0, img_height - (bbox[3] + expansion) / resolution),  # y-axis inverted
        min(img_height, img_height - (bbox[2] - expansion) / resolution),  # y-axis inverted
    ]
    crop_img = map_mask[int(bbox_img[2]):int(bbox_img[3]), int(bbox_img[0]):int(bbox_img[1])]
    extent = (bbox_img[0]*resolution, bbox_img[1]*resolution,
              (img_height - bbox_img[3])*resolution, (img_height - bbox_img[2])*resolution)

    return crop_img, extent

def plot_cropped_map(bbox, map_mask, resolution=0.1, expansion=10.0, ax=None):
    """
    Plot the cropped map around the bounding box.

    Args:
        bbox (list): Bounding box [xmin, xmax, ymin, ymax] in meters.
        map_mask (np.ndarray): The map mask image with shape (H, W).
        resolution (float): Resolution of the map in meters per pixel (m/px).
        expansion (float): Expansion in meters around the bounding box (meters).
        ax (matplotlib.axes.Axes): Matplotlib Axes to plot on. If None, creates a new figure.
    """
    if ax is None:
        ax = plt.gca()
    crop_img, extent = _get_cropped_map(bbox, map_mask, resolution, expansion)
    ax.imshow(crop_img, cmap='gray', extent=extent)
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')

def plot_nodelines_on_map(node_points, map_mask, resolution=0.1, expansion=10.0,
                          label=None, ax=None, **kwargs):
    """
    Plot lines connecting the given node points on the map.

    Args:
        node_points (list of tuple): List of (x, y) coordinates of the nodes.
        map_mask (np.ndarray): The map mask image with shape (H, W).
        resolution (float): Resolution of the map in meters per pixel (m/px).
        expansion (float): Expansion in meters around the bounding box (meters).
        label (str): Optional label for the plot.
        ax (matplotlib.axes.Axes): Matplotlib Axes to plot on. If None, creates a new figure.
        kwargs: Additional keyword arguments for Axes.plot().
    """
    if ax is None:
        ax = plt.gca()

    # Apply default styling if not provided
    if 'color' not in kwargs:
        kwargs['color'] = 'red'

    # Get bounding box of the node points
    bbox = np.min(node_points, axis=0).tolist() + np.max(node_points, axis=0).tolist()  # [xmin, ymin, xmax, ymax]
    bbox = [bbox[0], bbox[2], bbox[1], bbox[3]]  # Convert to [xmin, xmax, ymin, ymax]
    print('Bounding box of the linestring nodes:', bbox)
    # Convert node and bbox to image coordinates
    plot_cropped_map(bbox, map_mask, resolution, expansion, ax)
    # Plot nodes
    node_xs, node_ys = zip(*node_points)
    ax.plot(node_xs, node_ys, 'ro-', label=label, **kwargs)

# Plot exterior node lines
node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
               for token in first_polygon['exterior_node_tokens']]
plot_nodelines_on_map(node_points, map_mask)
plt.title('Polygon Exterior Nodes')

Polygons with holes

In [ ]:
# Find polygons with holes
polygons_with_holes = [p for p in polygons if len(p['holes']) > 0]
print('Number of polygons with holes:', len(polygons_with_holes))
print('First polygon with holes:', polygons_with_holes[-1])

### lines

In [ ]:
# Lines
lines = map_data['line']
print('Number of lines:', len(lines))
print('First line data:', lines[0])

# Number of nodes per line
nodes_per_line = [len(line['node_tokens']) for line in lines]
print('Average number of nodes per line:', np.mean(nodes_per_line))
print('Max number of nodes per line:', np.max(nodes_per_line))
print('Median number of nodes per line:', np.median(nodes_per_line))

### nodes

In [ ]:
# Nodes
nodes = map_data['node']
print('Number of nodes:', len(nodes))
print('First node data:', nodes[0])

### drivable_area

In [ ]:
# drivable_area
drivable_areas = map_data['drivable_area']
print('Number of drivable areas:', len(drivable_areas))
print('First drivable area data:', drivable_areas[0])

In [ ]:
# drivable area stats
from shapely.geometry import Polygon
# Number of polygons per drivable area
polygons_per_drivable_area = [len(da['polygon_tokens']) for da in drivable_areas]
print('Average number of polygons per drivable area:', np.mean(polygons_per_drivable_area))
print('Max number of polygons per drivable area:', np.max(polygons_per_drivable_area))
print('Median number of polygons per drivable area:', np.median(polygons_per_drivable_area))
# Area of polygons in drivable areas
polygons_in_drivable_areas = [
    [
        Polygon(  # The polygon is automatically closed by Shapely
            shell=[(nodes_dict[tk]['x'], nodes_dict[tk]['y']) 
                   for tk in polygons_dict[poly_token]['exterior_node_tokens']],
            holes=[
                [
                    (nodes_dict[tk]['x'], nodes_dict[tk]['y'])
                    for tk in hole['node_tokens']
                ] for hole in polygons_dict[poly_token]['holes']
            ]
        )
        for poly_token in da['polygon_tokens']
    ] for da in drivable_areas
]
areas_of_drivable_areas = [sum(p.area for p in polys) for polys in polygons_in_drivable_areas]
print('Minimum area of drivable areas (m^2):', np.min(areas_of_drivable_areas))
print('Average area of drivable areas (m^2):', np.mean(areas_of_drivable_areas))
print('Max area of drivable areas (m^2):', np.max(areas_of_drivable_areas))
print('Median area of drivable areas (m^2):', np.median(areas_of_drivable_areas))
# Areas of each polygon
polygon_areas = [poly.area for polys in polygons_in_drivable_areas for poly in polys]
print('Minimum area of polygons in drivable areas (m^2):', np.min(polygon_areas))
print('Average area of polygons in drivable areas (m^2):', np.mean(polygon_areas))
print('Max area of polygons in drivable areas (m^2):', np.max(polygon_areas))
print('Median area of polygons in drivable areas (m^2):', np.median(polygon_areas))
# Histogram of areas of polygons
plt.hist(polygon_areas, bins=20, color='green', alpha=0.7)
plt.title('Histogram of Polygon Sizes in Drivable Areas')
plt.xlabel('Area (m^2)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Show the biggest polygon of the first drivable area on the map
from matplotlib.path        import Path
from matplotlib.patches     import PathPatch

def plot_polygon(exterior_nodes, hole_nodes,
                 label=None, ax=None,
                 **kwargs):
    """
    Plot polygon.

    Args:
        exterior_nodes (list of tuple): List of (x, y) coordinates of the polygon exterior nodes.
        hole_nodes (list of list of tuple): List of holes, each hole is a list of (x, y) coordinates of the nodes.
        label (str): Label for the polygon.
        ax (matplotlib.axes.Axes): Matplotlib Axes to plot on. If None, creates a new figure.
        **kwargs: Additional keyword arguments passed to PathPatch.
    """
    if ax is None:
        ax = plt.gca()

    # Apply default styling if not provided
    if kwargs.get('facecolor') is None:
        kwargs['facecolor'] = 'red'  # Transparent fill by default
    if kwargs.get('edgecolor') is None:
        kwargs['edgecolor'] = 'darkred'  # Default edge color
    if kwargs.get('alpha') is None:
        kwargs['alpha'] = 0.5  # Default transparency
    if kwargs.get('linewidth') is None:
        kwargs['linewidth'] = 1  # Default line width
    
    verts = exterior_nodes + [exterior_nodes[0]]  # Close the polygon
    path_exterior = Path(np.asarray(verts))
    path_holes = [Path(np.asarray(hole + [hole[0]])) for hole in hole_nodes]
    path = Path.make_compound_path(path_exterior, *path_holes)
    patch = PathPatch(path, label=label, **kwargs)
    ax.add_patch(patch)

def plot_polygon_on_map(exterior_nodes, hole_nodes, map_mask,
                        resolution=0.1, expansion=10.0, bbox=None,
                        label=None, ax=None,
                        **kwargs):
    """
    Plot polygon over the map mask.

    Args:
        exterior_nodes (list of tuple): List of (x, y) coordinates of the polygon exterior nodes.
        hole_nodes (list of list of tuple): List of holes, each hole is a list of (x, y) coordinates of the nodes.
        map_mask (np.ndarray): The map mask image with shape (H, W).
        resolution (float): Resolution of the map in meters per pixel (m/px).
        expansion (float): Expansion in meters around the bounding box (meters)
        bbox (list): Bounding box [xmin, xmax, ymin, ymax] of the displayed area in meters. If None, computed from exterior nodes.
        label (str): Label for the polygon.
        ax (matplotlib.axes.Axes): Matplotlib Axes to plot on. If None, creates a new figure.
        **kwargs: Additional keyword arguments passed to PathPatch.

    Returns:
        ax (matplotlib.axes.Axes): The Axes with the plotted polygon.
    """
    if ax is None:
        ax = plt.gca()

    # Get bounding box of the node points
    if bbox is None:
        bbox = np.min(exterior_nodes, axis=0).tolist() + np.max(exterior_nodes, axis=0).tolist()  # [xmin, ymin, xmax, ymax]
        bbox = [bbox[0], bbox[2], bbox[1], bbox[3]]  # Convert to [xmin, xmax, ymin, ymax]
        print('Bounding box of the polygon exterior nodes:', bbox)
    # Plot cropped map
    print('Cropping and plotting map...')
    plot_cropped_map(bbox, map_mask, resolution, expansion, ax)
    # Plot polygon
    print('Plotting polygon on map...')
    plot_polygon(exterior_nodes, hole_nodes,
                 label=label, ax=ax, **kwargs)
    if label is not None:
        ax.legend()

    return ax

# Extract the biggest polygon in the first drivable area
first_drivable_area = drivable_areas[0]
polygon_indices_sorted_by_area = np.argsort([poly.area for poly in polygons_in_drivable_areas[0]])
biggest_polygon_token = first_drivable_area['polygon_tokens'][polygon_indices_sorted_by_area[-1]]
biggest_polygon = polygons_dict[biggest_polygon_token]
# Get node points of the exterior and holes
exterior_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
               for token in biggest_polygon['exterior_node_tokens']]
hole_node_points = [
    [
        (nodes_dict[token]['x'], nodes_dict[token]['y'])
        for token in hole['node_tokens']
    ] for hole in biggest_polygon['holes']
]
plot_polygon_on_map(exterior_node_points, hole_node_points, map_mask,
                    label='polygon in drivable area')
plt.title('Biggest Polygon in First Drivable Area')

### road_segment

In [ ]:
# road_segment
road_segments = map_data['road_segment']
print('Number of road segments:', len(road_segments))
print('First road segment data:', road_segments[0])

In [ ]:
# road_segment stats
road_segments_with_drivable_area = [rs for rs in road_segments if rs['drivable_area_token'] != '']
print('Number of road_segments with drivable areas:', len(road_segments_with_drivable_area))
road_segments_of_intersection = [rs for rs in road_segments if rs['is_intersection']]
print('Number of road_segments of intersections:', len(road_segments_of_intersection))

In [ ]:
# Plot first to fifth road_segments on map
indices = np.random.default_rng(SEED).choice(len(road_segments), size=NUM_DISP, replace=False)
for i in indices:
    road_segment = road_segments[i]
    segment_polygon = polygons_dict[road_segment['polygon_token']]
    # Get node points of the exterior and holes
    exterior_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                   for token in segment_polygon['exterior_node_tokens']]
    hole_node_points = [
        [
            (nodes_dict[token]['x'], nodes_dict[token]['y'])
            for token in hole['node_tokens']
        ] for hole in segment_polygon['holes']
    ]
    plot_polygon_on_map(exterior_node_points, hole_node_points, map_mask)
    plt.title(f'Road Segment {i+1}')
    plt.show()

### road_block

In [ ]:
# road_block
road_blocks = map_data['road_block']
print('Number of road blocks:', len(road_blocks))
print('First road block data:', road_blocks[0])

In [ ]:
# road_block stats
road_blocks_with_from_edge = [rb for rb in road_blocks if rb['from_edge_line_token'] != '']
print('Number of road_blocks with from_edge_line_token:', len(road_blocks_with_from_edge))
road_blocks_with_to_edge = [rb for rb in road_blocks if rb['to_edge_line_token'] != '']
print('Number of road_blocks with to_edge_line_token:', len(road_blocks_with_to_edge))
road_blocks_with_road_segment = [rb for rb in road_blocks if rb['road_segment_token'] != '']
print('Number of road_blocks with road_segment_token:', len(road_blocks_with_road_segment))

In [ ]:
# Plot first to fifth road_block on map
indices = np.random.default_rng(SEED).choice(len(road_blocks), size=NUM_DISP, replace=False)
for i in indices:
    road_block = road_blocks[i]
    block_polygon = polygons_dict[road_block['polygon_token']]
    # Get node points of the exterior and holes
    exterior_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                for token in block_polygon['exterior_node_tokens']]
    hole_node_points = [
        [
            (nodes_dict[token]['x'], nodes_dict[token]['y'])
            for token in hole['node_tokens']
        ] for hole in block_polygon['holes']
    ]
    # Get node points of the exterior and holes of the road_segment
    road_segment = road_segments_dict[road_block['road_segment_token']]
    road_segment_polygon = polygons_dict[road_segment['polygon_token']]
    road_segment_exterior_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                for token in road_segment_polygon['exterior_node_tokens']]
    road_segment_hole_node_points = [
        [
            (nodes_dict[token]['x'], nodes_dict[token]['y'])
            for token in hole['node_tokens']
        ] for hole in road_segment_polygon['holes']
    ]
    # Plot road_segment polygon
    plot_polygon_on_map(road_segment_exterior_node_points, road_segment_hole_node_points, map_mask,
                        label='road_segment')
    # Plot road_block polygon
    plot_polygon(exterior_node_points, hole_node_points, label='road_block',
                facecolor='purple')
    # Plot from_edge_line
    from_edge_nodes = lines_dict[road_block['from_edge_line_token']]['node_tokens']
    from_edge_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                            for token in from_edge_nodes]
    from_x, from_y = zip(*from_edge_node_points)
    plt.plot(from_x, from_y, 'bo-', label='from_edge_line')
    # Plot to_edge_line
    to_edge_nodes = lines_dict[road_block['to_edge_line_token']]['node_tokens']
    to_edge_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                        for token in to_edge_nodes]
    to_x, to_y = zip(*to_edge_node_points)
    plt.plot(to_x, to_y, 'go-', label='to_edge_line')
    plt.legend()
    plt.title(f'Road Block {i+1}')
    plt.show()

### lane

In [ ]:
# lane
lanes = map_data['lane']
print('Number of lanes:', len(lanes))
print('First lane data:', lanes[0])

In [ ]:
# lane stats
from collections import Counter

lanes_with_from_edge = [lane for lane in lanes if lane['from_edge_line_token'] != '']
print('Number of lanes with from_edge_line_token:', len(lanes_with_from_edge))
lanes_with_to_edge = [lane for lane in lanes if lane['to_edge_line_token'] != '']
print('Number of lanes with to_edge_line_token:', len(lanes_with_to_edge))
lanes_with_left_divider = [lane for lane in lanes if len(lane['left_lane_divider_segments']) > 0]
print('Number of lanes with left_lane_divider_segments:', len(lanes_with_left_divider))
lanes_with_right_divider = [lane for lane in lanes if len(lane['right_lane_divider_segments']) > 0]
print('Number of lanes with right_lane_divider_segments:', len(lanes_with_right_divider))
lanes_with_both_dividers = [lane for lane in lanes if len(lane['left_lane_divider_segments']) > 0 and len(lane['right_lane_divider_segments']) > 0]
print('Number of lanes with both left and right lane divider segments:', len(lanes_with_both_dividers))
# Count by lane type
lane_types = [lane['lane_type'] for lane in lanes]
lane_type_counts = Counter(lane_types)
print('Lane type counts:', lane_type_counts)

In [ ]:
# Plot first to fifth lane with both dividers on map
def plot_lane(lane, polygons_dict, lines_dict, nodes_dict, map_mask, ax=None):
    if ax is None:
        ax = plt.gca()

    lane_polygon = polygons_dict[lane['polygon_token']]
    # Get node points of the exterior and holes
    exterior_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                for token in lane_polygon['exterior_node_tokens']]
    hole_node_points = [
        [
            (nodes_dict[token]['x'], nodes_dict[token]['y'])
            for token in hole['node_tokens']
        ] for hole in lane_polygon['holes']
    ]
    # Plot lane polygon
    plot_polygon_on_map(exterior_node_points, hole_node_points, map_mask,
                        label='lane')
    # Plot from_edge_line
    from_edge_nodes = lines_dict[lane['from_edge_line_token']]['node_tokens']
    from_edge_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                            for token in from_edge_nodes]
    from_x, from_y = zip(*from_edge_node_points)
    plt.plot(from_x, from_y, 'bo-', label='from_edge_line')
    # Plot to_edge_line
    to_edge_nodes = lines_dict[lane['to_edge_line_token']]['node_tokens']
    to_edge_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                        for token in to_edge_nodes]
    to_x, to_y = zip(*to_edge_node_points)
    plt.plot(to_x, to_y, 'go-', label='to_edge_line')
    # Plot left_lane_divider
    left_divider_nodes = [seg['node_token'] for seg in lane['left_lane_divider_segments']]
    left_divider_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                                for token in left_divider_nodes]
    left_x, left_y = zip(*left_divider_node_points)
    plt.plot(left_x, left_y, 'cyan', marker='o', linestyle='-', label='left_lane_divider')
    # Plot right_lane_divider
    right_divider_nodes = [seg['node_token'] for seg in lane['right_lane_divider_segments']]
    right_divider_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                                for token in right_divider_nodes]
    right_x, right_y = zip(*right_divider_node_points)
    plt.plot(right_x, right_y, 'magenta', marker='o', linestyle='-', label='right_lane_divider')

    return ax

indices = np.random.default_rng(SEED).choice(len(lanes_with_both_dividers), size=NUM_DISP, replace=False)
for i in indices:
    lane = lanes_with_both_dividers[i]
    plot_lane(lane, polygons_dict, lines_dict, nodes_dict, map_mask)
    plt.legend()
    plt.title(f'Lane {i+1} with Both Dividers')
    plt.show()

### road_divider

In [ ]:
# road_divider
road_dividers = map_data['road_divider']
print('Number of road_dividers:', len(road_dividers))
print('First road_divider data:', road_dividers[0])

In [ ]:
# road_divider stats
from shapely.geometry import LineString

road_dividers_with_road_segment = [rd for rd in road_dividers if rd['road_segment_token'] != '']
print('Number of road_dividers with road_segment_token:', len(road_dividers_with_road_segment))
# Length of lines in road_dividers
linestrings_in_road_dividers = [
    LineString(
        [(nodes_dict[tk]['x'], nodes_dict[tk]['y'])
         for tk in lines_dict[rd['line_token']]['node_tokens']],
    ) for rd in road_dividers
]
lengths_of_road_dividers = [ls.length for ls in linestrings_in_road_dividers]
print('Minimum length of road_dividers (m):', np.min(lengths_of_road_dividers))
print('Average length of road_dividers (m):', np.mean(lengths_of_road_dividers))
print('Max length of road_dividers (m):', np.max(lengths_of_road_dividers))
print('Median length of road_dividers (m):', np.median(lengths_of_road_dividers))

In [ ]:
# Plot first to fifth road_dividers on map
indices = np.random.default_rng(SEED).choice(len(road_dividers_with_road_segment), size=NUM_DISP, replace=False)
for i in indices:
    road_divider = road_dividers_with_road_segment[i]
    road_divider_line = lines_dict[road_divider['line_token']]
    road_segment = road_segments_dict[road_divider['road_segment_token']]
    # Get node points of the exterior and holes of the road_segment
    road_segment_polygon = polygons_dict[road_segment['polygon_token']]
    road_segment_exterior_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                for token in road_segment_polygon['exterior_node_tokens']]
    road_segment_hole_node_points = [
        [
            (nodes_dict[token]['x'], nodes_dict[token]['y'])
            for token in hole['node_tokens']
        ] for hole in road_segment_polygon['holes']
    ]
    # Plot road_segment polygon
    plot_polygon_on_map(road_segment_exterior_node_points, road_segment_hole_node_points, map_mask,
                        label='road_segment')
    # Plot road_divider line
    road_divider_nodes = road_divider_line['node_tokens']
    road_divider_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                                for token in road_divider_nodes]
    road_divider_x, road_divider_y = zip(*road_divider_node_points)
    plt.plot(road_divider_x, road_divider_y, 'bo-', label='road_divider')
    plt.legend()
    plt.title(f'Road Divider {i+1}')
    plt.show()

### lane_divider

In [ ]:
# lane_divider
lane_dividers = map_data['lane_divider']
print('Number of lane_dividers:', len(lane_dividers))
print('First lane_divider data:', lane_dividers[0])

In [ ]:
# lane_divider stats
lane_dividers_with_segments = [ld for ld in lane_dividers if len(ld['lane_divider_segments']) > 0]
print('Number of lane_dividers with segments:', len(lane_dividers_with_segments))

In [ ]:
# Plot first to fifth lane_divider on map
indices = np.random.default_rng(SEED).choice(len(lane_dividers), size=NUM_DISP, replace=False)
for i in indices:
    lane_divider = lane_dividers[i]
    lane_divider_line = lines_dict[lane_divider['line_token']]
    # Plot lane_divider line
    lane_divider_nodes = lane_divider_line['node_tokens']
    lane_divider_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                                for token in lane_divider_nodes]
    plot_nodelines_on_map(lane_divider_node_points, map_mask, label='lane_divider')
    plt.legend()
    plt.title(f'Lane Divider {i+1}')
    plt.show()

### ped_crossing

In [ ]:
# ped_crossing
ped_crossings = map_data['ped_crossing']
print('Number of ped_crossings:', len(ped_crossings))
print('First ped_crossing data:', ped_crossings[0])

In [ ]:
# ped_crossing stats
ped_crossings_with_road_segment = [pc for pc in ped_crossings if pc['road_segment_token'] != '']
print('Number of ped_crossings with road_segment_token:', len(ped_crossings_with_road_segment))

In [ ]:
# Plot first to fifth ped_crossings on map
indices = np.random.default_rng(SEED).choice(len(ped_crossings), size=NUM_DISP, replace=False)
for i in indices:
    ped_crossing = ped_crossings[i]
    ped_crossing_polygon = polygons_dict[ped_crossing['polygon_token']]
    # Get node points of the exterior and holes
    exterior_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                for token in ped_crossing_polygon['exterior_node_tokens']]
    hole_node_points = [
        [
            (nodes_dict[token]['x'], nodes_dict[token]['y'])
            for token in hole['node_tokens']
        ] for hole in ped_crossing_polygon['holes']
    ]
    # Get node points of the exterior and holes of the road_segment
    road_segment = road_segments_dict[ped_crossing['road_segment_token']]
    road_segment_polygon = polygons_dict[road_segment['polygon_token']]
    road_segment_exterior_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                for token in road_segment_polygon['exterior_node_tokens']]
    road_segment_hole_node_points = [
        [
            (nodes_dict[token]['x'], nodes_dict[token]['y'])
            for token in hole['node_tokens']
        ] for hole in road_segment_polygon['holes']
    ]
    # Plot road_segment polygon
    plot_polygon_on_map(road_segment_exterior_node_points, road_segment_hole_node_points, map_mask,
                        label='road_segment')
    # Plot ped_crossing polygon
    plot_polygon(exterior_node_points, hole_node_points, label='ped_crossing',
                 facecolor='purple')
    plt.legend()
    plt.title(f'Ped Crossing {i+1}')
    plt.show()

### walkway

In [ ]:
# walkway
walkways = map_data['walkway']
print('Number of walkways:', len(walkways))
print('First walkway data:', walkways[0])

In [ ]:
# Plot first to fifth walkways on map
indices = np.random.default_rng(SEED).choice(len(walkways), size=NUM_DISP, replace=False)
for i in indices:
    walkway = walkways[i]
    walkway_polygon = polygons_dict[walkway['polygon_token']]
    # Get node points of the exterior and holes
    exterior_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                for token in walkway_polygon['exterior_node_tokens']]
    hole_node_points = [
        [
            (nodes_dict[token]['x'], nodes_dict[token]['y'])
            for token in hole['node_tokens']
        ] for hole in walkway_polygon['holes']
    ]
    # Plot walkway polygon
    plot_polygon_on_map(exterior_node_points, hole_node_points, map_mask,
                        label='walkway')
    plt.legend()
    plt.title(f'Walkway {i+1}')
    plt.show()

### stop_line

In [ ]:
# stop_line
stop_lines = map_data['stop_line']
print('Number of stop lines:', len(stop_lines))
print('First stop line data:', stop_lines[0])

In [ ]:
# stop_line stats
stop_lines_with_ped_crossing = [sl for sl in stop_lines if len(sl['ped_crossing_tokens']) > 0]
print('Number of stop_lines with ped_crossing_tokens:', len(stop_lines_with_ped_crossing))
stop_lines_with_traffic_light = [sl for sl in stop_lines if len(sl['traffic_light_tokens']) > 0]
print('Number of stop_lines with traffic_light_tokens:', len(stop_lines_with_traffic_light))
stop_lines_with_road_block = [sl for sl in stop_lines if sl['road_block_token'] != '']
print('Number of stop_lines with road_block_token:', len(stop_lines_with_road_block))
stop_lines_with_both = [sl for sl in stop_lines if len(sl['ped_crossing_tokens']) > 0 and len(sl['traffic_light_tokens']) > 0]
print('Number of stop_lines with both ped_crossing_tokens and traffic_light_tokens:', len(stop_lines_with_both))
# Number of traffic_lights in stop_lines
num_tl_in_stop_lines = [len(sl['traffic_light_tokens']) for sl in stop_lines]
print('Minimum number of traffic_lights in stop_lines:', np.min(num_tl_in_stop_lines))
print('Average number of traffic_lights in stop_lines:', np.mean(num_tl_in_stop_lines))
print('Max number of traffic_lights in stop_lines:', np.max(num_tl_in_stop_lines))
print('Median number of traffic_lights in stop_lines:', np.median(num_tl_in_stop_lines))

In [ ]:
# Plot first to fifth stop_lines on map
indices = np.random.default_rng(SEED).choice(len(stop_lines), size=NUM_DISP, replace=False)
for i in indices:
    stop_line = stop_lines[i]
    stop_line_polygon = polygons_dict[stop_line['polygon_token']]
    # Get node points of the exterior and holes
    exterior_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                for token in stop_line_polygon['exterior_node_tokens']]
    hole_node_points = [
        [
            (nodes_dict[token]['x'], nodes_dict[token]['y'])
            for token in hole['node_tokens']
        ] for hole in stop_line_polygon['holes']
    ]
    # Get node points of the exterior and holes of the road_block
    if stop_line['road_block_token'] != '':
        road_block = road_blocks_dict[stop_line['road_block_token']]
        road_block_polygon = polygons_dict[road_block['polygon_token']]
        road_block_exterior_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                    for token in road_block_polygon['exterior_node_tokens']]
        road_block_hole_node_points = [
            [
                (nodes_dict[token]['x'], nodes_dict[token]['y'])
                for token in hole['node_tokens']
            ] for hole in road_block_polygon['holes']
        ]
        # Plot road_block polygon
        plot_polygon_on_map(road_block_exterior_node_points, road_block_hole_node_points, map_mask,
                            label='road_block')
        # Plot stop_line polygon
        plot_polygon(exterior_node_points, hole_node_points, label='stop_line',
                     facecolor='purple', alpha=0.5)
    else:
        # Plot stop_line polygon
        plot_polygon_on_map(exterior_node_points, hole_node_points, map_mask,
                            facecolor='purple', label='stop_line')
    # Plot ped_crossings
    for pc_token in stop_line['ped_crossing_tokens']:
        ped_crossing = ped_crossings_dict[pc_token]
        ped_crossing_polygon = polygons_dict[ped_crossing['polygon_token']]
        # Get node points of the exterior and holes
        pc_exterior_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                    for token in ped_crossing_polygon['exterior_node_tokens']]
        pc_hole_node_points = [
            [
                (nodes_dict[token]['x'], nodes_dict[token]['y'])
                for token in hole['node_tokens']
            ] for hole in ped_crossing_polygon['holes']
        ]
        plot_polygon(pc_exterior_node_points, pc_hole_node_points,
                     label='ped_crossing', facecolor='orange', alpha=0.5)
    # Plot traffic_lights
    tl_locations = np.array([
        [traffic_light_dict[tl_token]['pose']['tx'], traffic_light_dict[tl_token]['pose']['ty']]
        for tl_token in stop_line['traffic_light_tokens']
    ])
    if len(tl_locations) > 0:
        plt.scatter(tl_locations[:, 0], tl_locations[:, 1], c='green', marker='x', s=100, label='traffic_light')
    
    plt.legend()
    plt.title(f'Stop Line {i+1}')
    plt.show()

### carpark_area

In [ ]:
# carpark_area
carpark_areas = map_data['carpark_area']
print('Number of carpark_areas:', len(carpark_areas))
print('First carpark_area data:', carpark_areas[0])

In [ ]:
# carpark_area stats
carpark_area_with_road_block = [pc for pc in carpark_areas if pc['road_block_token'] != '']
print('Number of carpark_areas with road_block_token:', len(carpark_area_with_road_block))

In [ ]:
# Plot first to fifth carpark_areas on map
indices = np.random.default_rng(SEED).choice(len(carpark_areas), size=NUM_DISP, replace=False)
for i in indices:
    carpark_area = carpark_areas[i]
    carpark_area_polygon = polygons_dict[carpark_area['polygon_token']]
    # Get node points of the exterior and holes
    exterior_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                for token in carpark_area_polygon['exterior_node_tokens']]
    hole_node_points = [
        [
            (nodes_dict[token]['x'], nodes_dict[token]['y'])
            for token in hole['node_tokens']
        ] for hole in carpark_area_polygon['holes']
    ]
    # Get node points of the exterior and holes of the road_block
    road_block = road_blocks_dict[carpark_area['road_block_token']]
    road_block_polygon = polygons_dict[road_block['polygon_token']]
    road_block_exterior_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                for token in road_block_polygon['exterior_node_tokens']]
    road_block_hole_node_points = [
        [
            (nodes_dict[token]['x'], nodes_dict[token]['y'])
            for token in hole['node_tokens']
        ] for hole in road_block_polygon['holes']
    ]
    # Plot road_block polygon
    plot_polygon_on_map(road_block_exterior_node_points, road_block_hole_node_points, map_mask,
                        label='road_block')
    # Plot ped_crossing polygon
    plot_polygon(exterior_node_points, hole_node_points, label='carpark_area',
                 facecolor='purple')
    plt.legend()
    plt.title(f'Carpark Area {i+1}')
    plt.show()

### traffic_light

In [ ]:
# traffic_light
traffic_lights = map_data['traffic_light']
print('Number of traffic_lights:', len(traffic_lights))
print('First traffic_light data:', traffic_lights[0])

In [ ]:
# traffic_light stats
from collections import Counter

traffic_lights_with_road_block = [tl for tl in traffic_lights if tl['from_road_block_token'] != '']
print('Number of traffic_lights with road_block_token:', len(traffic_lights_with_road_block))
# Count by traffic_light_type
traffic_light_types = [tl['traffic_light_type'] for tl in traffic_lights]
traffic_light_type_counts = Counter(traffic_light_types)
print('Traffic light type counts:', traffic_light_type_counts)

In [ ]:
# Plot first to fifth traffic_lights on map
indices = np.random.default_rng(SEED).choice(len(traffic_lights_with_road_block), size=NUM_DISP, replace=False)
for i in indices:
    traffic_light = traffic_lights_with_road_block[i]
    traffic_light_line = lines_dict[traffic_light['line_token']]
    road_block = road_blocks_dict[traffic_light['from_road_block_token']]
    # Get node points of the exterior and holes of the road_block
    road_block_polygon = polygons_dict[road_block['polygon_token']]
    road_block_exterior_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                for token in road_block_polygon['exterior_node_tokens']]
    road_block_hole_node_points = [
        [
            (nodes_dict[token]['x'], nodes_dict[token]['y'])
            for token in hole['node_tokens']
        ] for hole in road_block_polygon['holes']
    ]
    # Plot road_block polygon
    plot_polygon_on_map(road_block_exterior_node_points, road_block_hole_node_points, map_mask,
                        label='road_block')
    # Plot traffic_light line
    traffic_light_nodes = traffic_light_line['node_tokens']
    traffic_light_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                                for token in traffic_light_nodes]
    traffic_light_x, traffic_light_y = zip(*traffic_light_node_points)
    plt.plot(traffic_light_x, traffic_light_y, 'bo-', label='traffic_light_line')
    # Plot traffic_light location and direction
    tl_location = np.array([traffic_light_dict[traffic_light['token']]['pose']['tx'], 
                            traffic_light_dict[traffic_light['token']]['pose']['ty']])
    plt.scatter(tl_location[0], tl_location[1], c='red', marker='x', s=100, label='traffic_light_location')
    # Plot direction arrow
    tl_yaw = traffic_light_dict[traffic_light['token']]['pose']['rz']
    arrow_length = 5.0  # meters
    arrow_dx = arrow_length * np.cos(tl_yaw)
    arrow_dy = arrow_length * np.sin(tl_yaw)
    plt.arrow(tl_location[0], tl_location[1], arrow_dx, arrow_dy,
              head_width=1.0, head_length=1.5, fc='purple', ec='purple', label='traffic_light_direction')
    plt.legend()
    plt.title(f'Traffic Light {i+1}')
    plt.show()

### canvas_edge

In [ ]:
# canvas_edge
canvas_edges = map_data['canvas_edge']
print('canvas_edge:', canvas_edges)

### arcline_path_3

In [ ]:
# arcline_path_3
arcline_paths_3 = map_data['arcline_path_3']
print('Number of arcline_paths_3:', len(arcline_paths_3))
print('First arcline_path_3 data:', list(arcline_paths_3.values())[0])

In [ ]:
# arcline_path_3 stats
from collections import Counter

arcline_paths_3_multiple_segments = [ap for ap in arcline_paths_3.values() if len(ap) == 3]
print('Number of arcline_paths_3 with multiple segments:', len(arcline_paths_3_multiple_segments))
# Number of records in arcline_path_3
num_rec_in_arcline_path_3 = [len(v) for k, v in arcline_paths_3.items()]
print('Minimum number of records in arcline_path_3:', np.min(num_rec_in_arcline_path_3))
print('Average number of records in arcline_path_3:', np.mean(num_rec_in_arcline_path_3))
print('Max number of records in arcline_path_3:', np.max(num_rec_in_arcline_path_3))
print('Median number of records in arcline_path_3:', np.median(num_rec_in_arcline_path_3))
# Count by traffic_light_type
arcline_paths_3_shapes = [rec['shape'] for k, v in arcline_paths_3.items() for rec in v]
arcline_paths_3_shape_counts = Counter(arcline_paths_3_shapes)
print('Arcline paths 3 shape counts:', arcline_paths_3_shape_counts)

In [ ]:
from matplotlib.patches import Arc

def _calc_arc_params(start_pose, radius, turn_direction, arc_length):
    """
    Calculate the arc parameters.

    Args:
        pose (tuple): (x, y, yaw) coordinates of the pose.
        radius (float): Radius of the turn.
        turn_direction (str): 'L' for left turn, 'R' for right turn.
        arc_length (float): Length of the arc segment.

    Returns:
        center (tuple): (x, y) coordinates of the turning center.
        angle (float): Rotation angle of the arc in degrees for matplotlib.patches.Arc.
        theta1 (float): Starting angle of the arc in degrees for matplotlib.patches.Arc.
        theta2 (float): Ending angle of the arc in degrees for matplotlib.patches.Arc
        end_pose (tuple): (x, y, yaw) coordinates of the end pose after the arc.
    """
    # Caluculate the turning center
    x, y, yaw = start_pose
    if turn_direction == 'L':
        center_x = x + radius * np.cos(yaw + np.pi / 2)
        center_y = y + radius * np.sin(yaw + np.pi / 2)
    elif turn_direction == 'R':
        center_x = x + radius * np.cos(yaw - np.pi / 2)
        center_y = y + radius * np.sin(yaw - np.pi / 2)
    else:
        raise ValueError("turn_direction must be 'L' or 'R'")
    center = (center_x, center_y)
    # Calculate start and end angles
    angle = np.degrees(yaw - (np.pi/2 if turn_direction == 'L' else -np.pi/2))
    theta1 = 0 if turn_direction == 'L' else -np.degrees(arc_length/radius)
    theta2 = np.degrees(arc_length/radius) if turn_direction == 'L' else 0
    # Calculate end pose
    end_angle = angle + (theta2 if turn_direction == 'L' else theta1)
    end_x = center_x + radius * np.cos(np.radians(end_angle))
    end_y = center_y + radius * np.sin(np.radians(end_angle))
    end_yaw = np.radians(end_angle) + (np.pi/2 if turn_direction == 'L' else -np.pi/2)
    end_pose = (end_x, end_y, end_yaw)

    return center, angle, theta1, theta2, end_pose

def plot_dubins_path(start_pose, end_pose, shape, radius, segment_length,
                     label=None, ax=None,
                     **kwargs):
    """
    Plot Dubins path.

    Args:
        start_pose (tuple): (x, y, yaw) coordinates of the start pose.
        end_pose (tuple): (x, y, yaw) coordinates of the end pose.
        shape (str): Shape for the Dubins path. The supported shapes are 'LRL', 'RLR', 'LSL', 'LSR', 'RSL', 'RSR'.
        radius (float): Radius (meters) for the Dubins path.
        segment_length (tuple): (float, float, float) for the Dubins path segment lengths.
        label (str): Label for the polygon.
        ax (matplotlib.axes.Axes): Matplotlib Axes to plot on. If None, creates a new figure.
        **kwargs: Additional keyword arguments passed to matplotlib.patches.Arc and matplotlib.pyplot.plot.
    """
    if ax is None:
        ax = plt.gca()

    # Apply default styling if not provided
    if 'color' not in kwargs:
        kwargs['color'] = 'red'
    if 'linewidth' not in kwargs and 'lw' not in kwargs:
        kwargs['linewidth'] = 1

    # Plot the arrow of the start pose
    arrow_length = 1.0  # meters
    ax.arrow(start_pose[0], start_pose[1],
             arrow_length * np.cos(start_pose[2]), arrow_length * np.sin(start_pose[2]),
             head_width=0.5, head_length=0.5, fc=kwargs['color'], ec=kwargs['color'])

    first_mode, second_mode, third_mode = shape
    # Plot the first arc
    first_center, angle, theta1, theta2, second_pose = _calc_arc_params(start_pose, radius, first_mode, segment_length[0])
    first_arc = Arc(first_center, 2*radius, 2*radius,
                    angle=angle,
                    theta1=theta1,
                    theta2=theta2,
                    label=label,
                    **kwargs)
    ax.add_patch(first_arc)
    # Plot the second arc
    if second_mode in ['L', 'R']:  # Turn segment
        second_center, angle, theta1, theta2, third_pose = _calc_arc_params(second_pose, radius, second_mode, segment_length[1])
        second_arc = Arc(second_center, 2*radius, 2*radius,
                         angle=angle,
                         theta1=theta1,
                         theta2=theta2,
                         label=label,
                         **kwargs)
        ax.add_patch(second_arc)
    else:  # Straight line segment
        line_length = segment_length[1]
        line_end_x = second_pose[0] + line_length * np.cos(second_pose[2])
        line_end_y = second_pose[1] + line_length * np.sin(second_pose[2])
        ax.plot([second_pose[0], line_end_x], [second_pose[1], line_end_y],
                **kwargs)
        third_pose = (line_end_x, line_end_y, second_pose[2])
    # Plot the third arc
    third_center, angle, theta1, theta2, final_pose = _calc_arc_params(third_pose, radius, third_mode, segment_length[2])
    third_arc = Arc(third_center, 2*radius, 2*radius,
                    angle=angle,
                    theta1=theta1,
                    theta2=theta2,
                    **kwargs)
    ax.add_patch(third_arc)

def plot_arcline_path(arcline_path,
                      label=None, ax=None,
                      **kwargs):
    """
    Plot Dubins paths over the map mask.

    Args:
        arcline_path (list): List of Dubins path records. Each record is a dict with keys: 'start_pose', 'end_pose', 'shape', 'radius', 'segment_length'.
        label (str): Label for the arcpath. The shape is appended. The index is appended if multiple paths are plotted.
        ax (matplotlib.axes.Axes): Matplotlib Axes to plot on. If None, creates a new figure.
        **kwargs: Additional keyword arguments passed to matplotlib.patches.Arc.

    Returns:
        ax (matplotlib.axes.Axes): The Axes with the plotted polygon.
    """
    if ax is None:
        ax = plt.gca()
    
    if 'color' not in kwargs:
        color_list = ['red', 'blue', 'green', 'orange', 'purple']
    else:
        color_list = [kwargs['color']]
    
    # Extract parameters from arcline_path
    start_poses = [rec['start_pose'] for rec in arcline_path]
    end_poses = [rec['end_pose'] for rec in arcline_path]
    shapes = [rec['shape'] for rec in arcline_path]
    radiuses = [rec['radius'] for rec in arcline_path]
    segment_lengths = [rec['segment_length'] for rec in arcline_path]

    # Iterate through each Dubins path and plot
    for i, (start_pose, end_pose, shape, radius, segment_length) in enumerate(zip(start_poses, end_poses, shapes, radiuses, segment_lengths)):
        print('Plotting Dubins path on map...')
        # Plot settings
        if label is None:
            displayed_label = None
        elif len(start_poses) > 1:
            displayed_label = f"{label}_{i}_{shape}"
        else:
            displayed_label = f"{label}_{shape}"
        kwargs['color'] = color_list[i % len(color_list)]
        # Plot the Dubins path
        plot_dubins_path(start_pose, end_pose, shape, radius, segment_length,
                         label=displayed_label, ax=ax, **kwargs)

    return ax

def plot_arcline_path_on_map(arcline_path, map_mask,
                             resolution=0.1, expansion=10.0,
                             label=None, ax=None,
                             **kwargs):
    """
    Plot Dubins paths over the map mask.

    Args:
        arcline_path (list): List of Dubins path records. Each record is a dict with keys: 'start_pose', 'end_pose', 'shape', 'radius', 'segment_length'.
        map_mask (np.ndarray): The map mask image with shape (H, W).
        resolution (float): Resolution of the map in meters per pixel (m/px).
        expansion (float): Expansion in meters around the bounding box (meters)
        label (str): Label for the arcpath. The shape is appended. The index is appended if multiple paths are plotted.
        ax (matplotlib.axes.Axes): Matplotlib Axes to plot on. If None, creates a new figure.
        **kwargs: Additional keyword arguments passed to matplotlib.patches.Arc.

    Returns:
        ax (matplotlib.axes.Axes): The Axes with the plotted polygon.
    """
    if ax is None:
        ax = plt.gca()

    # Get bounding box of the node points
    start_poses = [rec['start_pose'] for rec in arcline_path]
    end_poses = [rec['end_pose'] for rec in arcline_path]
    nodes = [pose[:2] for pose in start_poses + end_poses]
    bbox = np.min(nodes, axis=0).tolist() + np.max(nodes, axis=0).tolist()  # [xmin, ymin, xmax, ymax]
    bbox = [bbox[0], bbox[2], bbox[1], bbox[3]]  # Convert to [xmin, xmax, ymin, ymax]
    print('Bounding box of the arcline:', bbox)
    # Plot cropped map
    print('Cropping and plotting map...')
    plot_cropped_map(bbox, map_mask, resolution, expansion, ax)
    # Plot the arcline path
    plot_arcline_path(arcline_path,
                      label=label, ax=ax,
                      **kwargs)
    if label is not None:
        ax.legend()

    return ax

# Plot first to fifth arcline_path_3 on map
indices = np.random.default_rng(SEED).choice(len(arcline_paths_3_multiple_segments), size=NUM_DISP, replace=False)
for i in indices:
    arcline_path = list(arcline_paths_3.values())[i]
    
    # Plot the Dubins path
    plot_arcline_path_on_map(arcline_path, map_mask,
                             label='arcline')
    plt.legend()
    plt.title(f'Arcline Path {i+1}')
    plt.show()

### connectivity

In [ ]:
# connectivity
connectivities = map_data['connectivity']
print('Number of connectivities:', len(connectivities))
print('First connectivity data:', list(connectivities.values())[0])

In [ ]:
# connectivity stats
connectivities_with_incoming = [v for k, v in connectivities.items() if v['incoming'] != '']
print('Number of connectivities with incoming:', len(connectivities_with_incoming))
connectivities_with_outgoing = [v for k, v in connectivities.items() if v['outgoing'] != '']
print('Number of connectivities with outgoing:', len(connectivities_with_outgoing))

### lane_connector

In [ ]:
# lane_connector
lane_connectors = map_data['lane_connector']
print('Number of lane_connectors:', len(lane_connectors))
print('First lane_connector data:', lane_connectors[0])

In [ ]:
# Plot first to fifth lane_connectors with their lanes on map
indices = np.random.default_rng(SEED).choice(len(lane_connectors), size=NUM_DISP, replace=False)
for i in indices:
    lane_connector = lane_connectors[i]
    lane_connector_polygon = polygons_dict[lane_connector['polygon_token']]
    # Get node points of the exterior and holes
    exterior_node_points = [(nodes_dict[token]['x'], nodes_dict[token]['y']) 
                for token in lane_connector_polygon['exterior_node_tokens']]
    hole_node_points = [
        [
            (nodes_dict[token]['x'], nodes_dict[token]['y'])
            for token in hole['node_tokens']
        ] for hole in lane_connector_polygon['holes']
    ]
    exterior_points = np.array(exterior_node_points)
    # Get incoming lanes
    incoming_lane_tokens = connectivities[lane_connector['token']]['incoming']
    incoming_lanes = [lanes_dict[lane_token] for lane_token in incoming_lane_tokens]
    incoming_lanes_exterior = [
        [
            (nodes_dict[token]['x'], nodes_dict[token]['y'])
            for token in polygons_dict[lane['polygon_token']]['exterior_node_tokens']
        ] for lane in incoming_lanes
    ]
    incoming_lanes_holes = [
        [
            [
                (nodes_dict[token]['x'], nodes_dict[token]['y'])
                for token in hole['node_tokens']
            ] for hole in polygons_dict[lane['polygon_token']]['holes']
        ] for lane in incoming_lanes
    ]
    exterior_points = np.vstack([exterior_points] + incoming_lanes_exterior)
    # Get outgoing lanes
    outgoing_lane_tokens = connectivities[lane_connector['token']]['outgoing']
    outgoing_lanes = [lanes_dict[lane_token] for lane_token in outgoing_lane_tokens]
    outgoing_lanes_exterior = [
        [
            (nodes_dict[token]['x'], nodes_dict[token]['y'])
            for token in polygons_dict[lane['polygon_token']]['exterior_node_tokens']
        ] for lane in outgoing_lanes
    ]
    outgoing_lanes_holes = [
        [
            [
                (nodes_dict[token]['x'], nodes_dict[token]['y'])
                for token in hole['node_tokens']
            ] for hole in polygons_dict[lane['polygon_token']]['holes']
        ] for lane in outgoing_lanes
    ]
    exterior_points = np.vstack([exterior_points] + outgoing_lanes_exterior)
    # Get bounding box of the node points
    bbox = np.min(exterior_points, axis=0).tolist() + np.max(exterior_points, axis=0).tolist()  # [xmin, ymin, xmax, ymax]
    bbox = [bbox[0], bbox[2], bbox[1], bbox[3]]  # Convert to [xmin, xmax, ymin, ymax]
    print('Bounding box of the lane_connector and its lanes:', bbox)
    # Plot lane_connector polygon
    plot_polygon_on_map(exterior_node_points, hole_node_points, map_mask,
                        bbox=bbox, label='lane_connector_polygon')
    # Plot lane_connector arcpath
    lane_connector_arcpath = arcline_paths_3[lane_connector['token']]
    plot_arcline_path(lane_connector_arcpath, label='lane_connector_arcpath')
    # Plot incoming lanes polygons and arpaths
    for ext_nodes, hole_nodes, token in zip(incoming_lanes_exterior, incoming_lanes_holes, incoming_lane_tokens):
        plot_polygon(ext_nodes, hole_nodes, label='incoming_lane', facecolor='cyan', alpha=0.5)
        incoming_lane_arcpath = arcline_paths_3[token]
        plot_arcline_path(incoming_lane_arcpath, label='incoming_lane_arcpath', color='blue')
    # Plot outgoing lanes polygons and arpaths
    for ext_nodes, hole_nodes, token in zip(outgoing_lanes_exterior, outgoing_lanes_holes, outgoing_lane_tokens):
        plot_polygon(ext_nodes, hole_nodes, label='outgoing_lane', facecolor='magenta', alpha=0.5)
        outgoing_lane_arcpath = arcline_paths_3[token]
        plot_arcline_path(outgoing_lane_arcpath, label='outgoing_lane_arcpath', color='purple')
    plt.legend()
    plt.title(f'Lane {i+1} with Both Dividers')
    plt.show()